In [8]:
!pip install pyproj
!pip install -U numpy==1.26.4
!pip install transformers
!pip install -U accelerate
!pip install pandas
!pip install PyPDF2
!pip install NLTK
!pip install editdistance
!pip install paramiko
!pip install func_timeout

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/alex/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/alex/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [1]:
# import libraries
import urllib.request
import requests
import json
from pyproj import Transformer
import os
import time
import sys
from datetime import datetime
import nltk
import re
import pandas as pd

# import common functions
sys.path.insert(1, '../')
import common

# set name of module, to fetch info from config
module_name = "sidestone"

/home/alex/anaconda3/envs/agnes-bert-file-import/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False
cuda gpu number is 0


Some weights of the model checkpoint at alexbrandsen/ArcheoBERTje-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/home/alex/anaconda3/envs/agnes-bert-file-import/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more 

In [6]:
# get info from config file
config = common.get_config()

# log config info        
pdf_folder = config['data_source'][module_name]['pdf_folder']
print(f'pdf_folder: {pdf_folder}')

json_folder = config['data_source'][module_name]['json_folder']
print(f'json_folder: {json_folder}')

html_folder = config['data_source'][module_name]['html_folder']
print(f'html_folder: {html_folder}')

language = config['data_source'][module_name]['language']
print(f'language: {language}')

bert_model = config['bert_models'][language]
print(f'bert_model: {bert_model}')



pdf_folder: /media/alex/Data/agnes_data/sidestone/pdf/
json_folder: /media/alex/Data/agnes_data/sidestone/json/
html_folder: /media/alex/Data/agnes_data/sidestone/html/
language: dutch
bert_model: /media/alex/Data/agnes_models/ArcheoBERTje-NER


In [3]:
def remove_html_tags(text):
    #Remove html tags from a string
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)



In [4]:

"""
CSV_extract extracts a .csv file from a webpage, filters it by language and keywords parameters, and automatically downloads .pdf files from it to a specific folder.
:param url: the url of the .csv
:param pdf_download_folder: the folder in which the downloaded .pdfs are stored.
:param pdf_download_prefix: the first part of the download url that all .pdf files share 
:return: a string of the header of the .csv file, and all newly downloaded .pdfs. This returns all the records that have been added to the .csv after the last run. 
"""
#extract .csv from url
# r = requests.get(url, allow_redirects=True)

# #cleans .csv and adds a line break
# cleanText = r.content.decode('utf-8')
# cleanText = cleanText.replace("\r", "").replace("\n", "").replace("\r", "").replace("\t", "")
# cleanText = cleanText.replace("<br />", "\n")


# with open("sidestone.csv", "w", encoding="utf-8") as f:
#     f.write(remove_html_tags(cleanText.replace("@", "\t"))) #remove html tags from .csv file, and writes cleaned string to a temporary .csv file


#- Converts .csv to a Pandas dataframe
df = pd.read_csv("sidestone-export-2025.csv")

# Filter parameters for the language and keywords columns:
lang_options = ["eng", "dut", "ger"]
key_options = ["archaeology", "archeologie", "archäologie", "archaeo", "archeo", "prehistory", "bronze age", "iron age", "lithic"]

NewDF = df.dropna(axis = 0, subset="Keywords") #drop all records with no keywords
print(NewDF)

#drop all records not written in languages in lang_options: 
for record in NewDF["Language"]:
    if not any(option in record for option in lang_options):
        NewDF.drop(NewDF.loc[NewDF["Language"]==record].index, inplace=True)

#drop all records without keywords that are in key_options:
for record in NewDF["Keywords"]:
    if not any(option in record.lower() for option in key_options):
        NewDF.drop(NewDF.loc[NewDF["Keywords"]==record].index, inplace=True)


#save filtered dataframe as a .csv to a temporary file:
NewDF.to_csv("sidestone-export-2025-filtered.csv", index=False)


# with open("sidestone-export-2025-filtered.csv", "r", encoding="utf-8") as f:
#     lines = f.read().splitlines()
#     not_yet_downloaded = [lines[0]] #header
    
#     for p in lines[1:]:
#         file_url = p[p.find(pdf_download_prefix):] #for each record, find the download url
#         file_name = file_url[32:]

#         if file_name in os.listdir(pdf_download_folder):
#             #print message for already downloaded file
#             print(str(file_name) + " already in " + str(pdf_download_folder))
#         else:
#             #download new record and save it as pdf:
#             r = requests.get(file_url, allow_redirects=True)
#             with open(str(pdf_download_folder) + "/" + str(file_name), 'wb') as b:
#                 b.write(r.content)
#             not_yet_downloaded.append(p) #create list of all newly downloaded files



                         Publisher         ISBN13  \
0                  Sidestone Press  9789464263534   
1                  Sidestone Press  9789464263541   
2                  Sidestone Press  9789464263503   
3                  Sidestone Press  9789464263510   
4    Sidestone Press Dissertations  9789464280753   
..                             ...            ...   
909  Sidestone Press Dissertations  9789464280975   
910  Sidestone Press Dissertations  9789464280999   
911  Sidestone Press Dissertations  9789464281002   
912      Sidestone Press Academics  9789464271201   
913      Sidestone Press Academics  9789464271218   

                                     DOI ISBN  EAN  \
0       https://doi.org/10.59641/zz090cl  NaN  NaN   
1       https://doi.org/10.59641/zz090cl  NaN  NaN   
2       https://doi.org/10.59641/xx812aj  NaN  NaN   
3       https://doi.org/10.59641/xx812aj  NaN  NaN   
4       https://doi.org/10.59641/x0922aj  NaN  NaN   
..                                   ..

/tmp/ipykernel_1068988/3059801534.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  NewDF.drop(NewDF.loc[NewDF["Language"]==record].index, inplace=True)
/tmp/ipykernel_1068988/3059801534.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  NewDF.drop(NewDF.loc[NewDF["Keywords"]==record].index, inplace=True)


In [12]:
docs_done = []

#processing the dataframe
for index, row in NewDF.iterrows():

    print('--------------------------------------------')
    print('Processing book: '+row['Title'])
        
    
    # generate document dictionary
    output_document = {}
    output_document['source'] = module_name
    #output_document['file_name'] = common.cleanFileName(dataFile['filename'])
    output_document['description'] = row['Main description']
    output_document['title'] = row['Title'] + ' ' + str(row['Subtitle'])
    creators = []
    for auteur in str(row['Surname, Name']).split(' | '):
        creators.append(auteur)
    output_document['creators'] = creators
    output_document['publisher'] = row['Publisher']
    output_document['createdAt'] = row['Publication date']
    
    url = f"https://www.sidestone.com/permlink/{row['ISBN13']}"
    identifiers = {
        'DOI':row['DOI'],
        'url':url,
        'ISBN':row['ISBN13'],
    }
    output_document['identifiers'] = identifiers
                    
    # set document identifier
    doc_id = f"{row['ISBN13']}"
    
    print(f"doc_id:{doc_id}")

    # # TEMP if already downloaded, skip
    # output_location = f"{pdf_folder}_{row['ISBN13']}.pdf"
    # if os.path.isfile(output_location):
    #     print('already downloaded, SKIPPING')
    #     continue


    # check if DOI already done
    if not row['Cover-URL'] in docs_done:
        
        # sometimes get download error, so try and continue if error
        try:
            # download pdf https://sidestone.com/downloads/9789464263534.pdf
            file_url =   f"https://sidestone.com/downloads/{row['ISBN13']}.pdf"
            print(f"trying download at url: {file_url}")
            pdf_location = common.downloaddocument(
                file_url, 
                '', 
                pdf_folder, 
                f"{row['ISBN13']}.pdf"
            )
            print(f"downloaded pdf")
            docs_done.append(row['Cover-URL'])
        except Exception as e:
            try:
                # download pdf
                file_url = f"https://sidestone.com/openaccess/{row['ISBN13']}.pdf"
                print(f"trying download at url: {file_url}")
                pdf_location = common.downloaddocument(
                    file_url, 
                    '', 
                    pdf_folder, 
                    f"{row['ISBN13']}.pdf"
                )
                print(f"downloaded pdf")
                docs_done.append(row['Cover-URL'])
            except Exception as e:
                print(f"could not download pdf, error:")
                print(e)
                continue
                
        # save document.json 
        json_output_folder = f"{json_folder}/{doc_id}"
        common.savejson(output_document, f"{json_output_folder}/document.json")
    
        print(f"saved doc json")
    
        # process pdf, store page.json files with entities 
        common.run_ner_on_pdf(
            pdf_location, 
            json_output_folder, 
            bert_model, 
            language
        )
    
        print(f"ran NER, saved page json")
    
        # process pdf, save html files
        html_output_folder = f"{html_folder}/{doc_id}"
        common.pdf2html(pdf_location, html_output_folder)
    
        print(f"generated and saved html")
    
        
          

                              

--------------------------------------------
Processing book: St. Eustatius
doc_id:9789464263534
trying download at url: https://sidestone.com/downloads/9789464263534.pdf
downloaded pdf
saved doc json
ran NER, saved page json
Converted /media/alex/Data/agnes_data/sidestone/pdf/_9789464263534.pdf to html
generated and saved html
--------------------------------------------
Processing book: St. Eustatius
doc_id:9789464263541
--------------------------------------------
Processing book: The Handle Core Concept
doc_id:9789464280753
trying download at url: https://sidestone.com/downloads/9789464280753.pdf
downloaded pdf
saved doc json
ran NER, saved page json
Converted /media/alex/Data/agnes_data/sidestone/pdf/_9789464280753.pdf to html
generated and saved html
--------------------------------------------
Processing book: The Handle Core Concept
doc_id:9789464280760
--------------------------------------------
Processing book: Identity, Power and Group Formation in Archaic Macedonia (600-40

Xref table not zero-indexed. ID numbers for objects will be corrected.


downloaded pdf
saved doc json
ran NER, saved page json
Converted /media/alex/Data/agnes_data/sidestone/pdf/_9789464262025.pdf to html
generated and saved html
--------------------------------------------
Processing book: Mentale Konzepte der Stadt in Bild- und Textmedien der Vormoderne
doc_id:9789464270570
trying download at url: https://sidestone.com/downloads/9789464270570.pdf
downloaded pdf
saved doc json
ran NER, saved page json
Converted /media/alex/Data/agnes_data/sidestone/pdf/_9789464270570.pdf to html
generated and saved html
--------------------------------------------
Processing book: Mentale Konzepte der Stadt in Bild- und Textmedien der Vormoderne
doc_id:9789464270587
--------------------------------------------
Processing book: Roots of Routes. Mobility and Networks between the Past and the Future
doc_id:9789464261912
trying download at url: https://sidestone.com/downloads/9789464261912.pdf
downloaded pdf
saved doc json
ran NER, saved page json
Converted /media/alex/Data/

Xref table not zero-indexed. ID numbers for objects will be corrected.


downloaded pdf
saved doc json
ran NER, saved page json
Converted /media/alex/Data/agnes_data/sidestone/pdf/_9789464261578.pdf to html
generated and saved html
--------------------------------------------
Processing book: Revealing Christian Heritage. Volume I
doc_id:9789464261585
--------------------------------------------
Processing book: Keramik jenseits von 'Kulturen'
doc_id:9789464280456
trying download at url: https://sidestone.com/downloads/9789464280456.pdf
downloaded pdf
saved doc json
ran NER, saved page json
Converted /media/alex/Data/agnes_data/sidestone/pdf/_9789464280456.pdf to html
generated and saved html
--------------------------------------------
Processing book: Keramik jenseits von 'Kulturen'
doc_id:9789464280463
--------------------------------------------
Processing book: Thirdspace in Assyrien und Urartu
doc_id:9789464280548
trying download at url: https://sidestone.com/downloads/9789464280548.pdf
trying download at url: https://sidestone.com/openaccess/97894642

...

downloaded pdf
saved doc json
ran NER, saved page json
Converted /media/alex/Data/agnes_data/sidestone/pdf/_9789082225105.pdf to html
generated and saved html
--------------------------------------------
Processing book: APL 2 - Collection of Papers
doc_id:9789060210680
trying download at url: https://sidestone.com/downloads/9789060210680.pdf


Xref table not zero-indexed. ID numbers for objects will be corrected.


downloaded pdf
saved doc json
ran NER, saved page json
Converted /media/alex/Data/agnes_data/sidestone/pdf/_9789060210680.pdf to html
generated and saved html
--------------------------------------------
Processing book: APL 1 - Collection of Papers
doc_id:9789060210673
trying download at url: https://sidestone.com/downloads/9789060210673.pdf


Xref table not zero-indexed. ID numbers for objects will be corrected.


downloaded pdf
saved doc json
ran NER, saved page json
Converted /media/alex/Data/agnes_data/sidestone/pdf/_9789060210673.pdf to html
generated and saved html
--------------------------------------------
Processing book: From Ros to Prut (volume 1)
doc_id:9789464270723
trying download at url: https://sidestone.com/downloads/9789464270723.pdf
trying download at url: https://sidestone.com/openaccess/9789464270723.pdf
could not download pdf, error:
HTTP Error 404: Not Found
--------------------------------------------
Processing book: From Ros to Prut (volume 1)
doc_id:9789464270730
trying download at url: https://sidestone.com/downloads/9789464270730.pdf
trying download at url: https://sidestone.com/openaccess/9789464270730.pdf
could not download pdf, error:
HTTP Error 404: Not Found
--------------------------------------------
Processing book: From Ros to Prut (volume 2)
doc_id:9789464270754
trying download at url: https://sidestone.com/downloads/9789464270754.pdf
trying download at url

In [14]:
         
# upload json and html to webserver
common.upload2webserver(json_folder, html_folder, module_name, config['webserver']['json_folder'], config['webserver']['html_folder'])

print(f"uploaded json/html to webserver")

# remotely start indexing script on webserver
common.start_index(module_name)

print(f"indexing on webserver started")

print(f"done!")



uploaded json/html to webserver
indexing on webserver started
done!
